In [3]:
# =============================================================================
# COLAB SETUP
# =============================================================================
from google.colab import files
uploaded = files.upload()   # Upload both CSVs here

# =============================================================================
# IMPORTS
# =============================================================================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix

# =============================================================================
# OUTPUT DIRECTORY
# =============================================================================
OUTPUT_DIR = "/content/outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save(fig, name):
    fig.savefig(f"{OUTPUT_DIR}{name}", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Saved {name}")

# =============================================================================
# LOAD DATA
# =============================================================================
fg  = pd.read_csv("fear_greed_index.csv")
raw = pd.read_csv("historical_data.csv")

# Normalize column names (IMPORTANT)
fg.columns = fg.columns.str.strip().str.lower()
raw.columns = raw.columns.str.strip().str.lower()

print("Fear/Greed columns:", fg.columns)
print("Trader data columns:", raw.columns)

# =============================================================================
# DATA CLEANING
# =============================================================================

# --- Fix column names dynamically ---
# Fear Greed
fg.rename(columns={
    "date": "date",
    "value": "value",
    "classification": "classification"
}, inplace=True)

# Trader dataset flexible mapping
raw.rename(columns={
    "account": "account",
    "symbol": "symbol",
    "execution price": "execution_price",
    "size": "size",
    "size usd": "size_usd",
    "side": "side",
    "time": "time",
    "timestamp ist": "time",
    "closedpnl": "closed_pnl",
    "closed pnl": "closed_pnl",
    "fee": "fee",
    "trade id": "trade_id"
}, inplace=True)

# Convert datetime
raw["datetime"] = pd.to_datetime(raw["time"], errors="coerce")
raw.dropna(subset=["datetime"], inplace=True)
raw["date"] = raw["datetime"].dt.normalize()

fg["date"] = pd.to_datetime(fg["date"]).dt.normalize()

# Convert numeric
for col in ["closed_pnl", "size_usd", "execution_price"]:
    if col in raw.columns:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

raw.dropna(subset=["closed_pnl", "size_usd"], inplace=True)

print("Cleaned rows:", len(raw))

# =============================================================================
# FEATURE ENGINEERING
# =============================================================================
raw["is_win"] = (raw["closed_pnl"] > 0).astype(int)
raw["is_long"] = raw["side"].str.lower().eq("buy")
raw["is_short"] = raw["side"].str.lower().eq("sell")

# Daily aggregation
daily = (
    raw.groupby(["account", "date"])
    .agg(
        daily_pnl=("closed_pnl", "sum"),
        n_trades=("closed_pnl", "count"),
        win_rate=("is_win", "mean"),
        avg_size_usd=("size_usd", "mean"),
        long_trades=("is_long", "sum"),
        short_trades=("is_short", "sum"),
    )
    .reset_index()
)

daily["long_short_ratio"] = daily["long_trades"] / (daily["short_trades"] + 1e-9)

# Merge
merged = daily.merge(fg[["date", "value", "classification"]], on="date", how="inner")

merged["sentiment"] = merged["classification"].apply(
    lambda x: "Fear" if "fear" in str(x).lower() else "Greed"
)

# =============================================================================
# ANALYSIS
# =============================================================================

def drawdown(x):
    return x[x < 0].mean()

perf = merged.groupby("sentiment").agg(
    avg_pnl=("daily_pnl", "mean"),
    win_rate=("win_rate", "mean"),
    drawdown=("daily_pnl", drawdown)
).reset_index()

print("\nPerformance:\n", perf)

# Plot 1
fig, ax = plt.subplots()
sns.barplot(data=perf, x="sentiment", y="avg_pnl", ax=ax)
ax.set_title("Avg PnL: Fear vs Greed")
save(fig, "pnl.png")

# =============================================================================
# BEHAVIOR
# =============================================================================
bhv = merged.groupby("sentiment").agg(
    trades=("n_trades", "mean"),
    size=("avg_size_usd", "mean")
).reset_index()

print("\nBehavior:\n", bhv)

# =============================================================================
# SEGMENTATION
# =============================================================================
acct = merged.groupby("account").agg(
    avg_pnl=("daily_pnl", "mean"),
    trades=("n_trades", "mean"),
    size=("avg_size_usd", "mean"),
    win_rate=("win_rate", "mean")
).reset_index()

acct["segment"] = pd.qcut(acct["trades"], 2, labels=["Low", "High"])

print("\nSegments:\n", acct.groupby("segment").mean(numeric_only=True))

# =============================================================================
# MODEL
# =============================================================================
daily_feat = merged.groupby("date").agg(
    pnl=("daily_pnl", "mean"),
    trades=("n_trades", "mean"),
    sentiment=("value", "mean")
).reset_index()

daily_feat["target"] = (daily_feat["pnl"].shift(-1) > 0).astype(int)
daily_feat.dropna(inplace=True)

X = daily_feat[["trades", "sentiment"]]
y = daily_feat["target"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nModel Report:\n", classification_report(y_test, y_pred))

# =============================================================================
# CLUSTERING
# =============================================================================
Xc = acct[["avg_pnl", "trades", "size", "win_rate"]].dropna()

scaler = StandardScaler()
Xc = scaler.fit_transform(Xc)

kmeans = KMeans(n_clusters=3, random_state=42)
acct["cluster"] = kmeans.fit_predict(Xc)

print("\nClusters:\n", acct.groupby("cluster").mean(numeric_only=True))

# =============================================================================
# SUMMARY FILE
# =============================================================================
summary = """
KEY INSIGHTS:
1. Greed days generally show higher PnL but larger losses.
2. Fear days lead to overtrading behavior.
3. High activity traders show mixed performance.

RECOMMENDATIONS:
- Reduce trading during Fear periods
- Control leverage during Greed
"""

with open(f"{OUTPUT_DIR}summary.txt", "w") as f:
    f.write(summary)

# =============================================================================
# DOWNLOAD OUTPUTS
# =============================================================================
shutil.make_archive("outputs", 'zip', OUTPUT_DIR)
files.download("outputs.zip")

print("\n✅ DONE - Outputs downloaded")


Saving historical_data.csv to historical_data.csv
Saving fear_greed_index.csv to fear_greed_index.csv
Fear/Greed columns: Index(['timestamp', 'value', 'classification', 'date'], dtype='object')
Trader data columns: Index(['account', 'coin', 'execution price', 'size tokens', 'size usd', 'side',
       'timestamp ist', 'start position', 'direction', 'closed pnl',
       'transaction hash', 'order id', 'crossed', 'fee', 'trade id',
       'timestamp'],
      dtype='object')
Cleaned rows: 79225

Performance:
   sentiment      avg_pnl  win_rate      drawdown
0      Fear  9387.502733  0.319269  -5402.866770
1     Greed  5415.243971  0.343044 -15207.805222
✓ Saved pnl.png

Behavior:
   sentiment     trades         size
0      Fear  85.236842  7539.752824
1     Greed  57.850000  6891.301065

Segments:
               avg_pnl      trades         size  win_rate
segment                                                 
Low       2146.112395   19.017056  8216.455693  0.335065
High     14560.780451  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ DONE - Outputs downloaded
